# Chapter 3 — The 4 Components of Every Great Prompt

---

### The Story So Far
You deployed a naive classifier on Monday. Wednesday morning — Slack exploded.
The AI was returning 120-word essays instead of category names.

**Root cause:** Zero prompt structure. Zero instructions.

**Fix:** Add the 4 components — Role, Instruction, Input, Output Format.

In this notebook, we fix it — one component at a time.

---

### 4 Tasks
- **Task 1:** Run V0 naive on 5 emails — see the chaos
- **Task 2:** Build V4 structured — fix the chaos  
- **Task 3:** Remove `temperature=0` — see non-determinism
- **Task 4:** Add Confidence Score — 5th component

## Setup — Run This First

In [3]:
from openai import OpenAI
import json
import time
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
client = OpenAI()

# Load test emails
with open("test_emails.json", "r") as f:
    emails = json.load(f)

# Pick 5 emails for this chapter
# Mix of easy + medium + hard
five_emails = [
    emails[0],   # easy billing
    emails[5],   # easy technical
    emails[10],  # easy feature request
    emails[22],  # medium — sarcasm
    emails[42],  # hard — multi-issue
]

print(f"Total emails in JSON : {len(emails)}")
print(f"Using for this chapter: 5 emails")
print()
for i, e in enumerate(five_emails):
    print(f"Email {i+1}: [{e['difficulty']:6}] {e['subject']}")

Total emails in JSON : 100
Using for this chapter: 5 emails

Email 1: [easy  ] Invoice Discrepancy
Email 2: [easy  ] Login Issues
Email 3: [easy  ] Integration with Slack
Email 4: [medium] Is the System Down?
Email 5: [hard  ] API Limits Inquiry


---
## Task 1 — V0 Naive Prompt
### What happens when you give AI zero instructions?

This is the exact 15 lines you wrote on Monday morning.
Run it on 5 emails. Document the chaos.

In [4]:
# ── V0: NAIVE — No components, no structure ──
def classify_v0(email):
    """
    The broken classifier from Monday morning.
    Prompt = just 1 line. No role. No format. No rules.
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": f"Classify this email: {email['body']}"}
        ]
    )
    return response.choices[0].message.content


# Run on all 5 emails
print("=" * 60)
print("V0 NAIVE CLASSIFIER — WEDNESDAY MORNING CHAOS")
print("=" * 60)

for i, email in enumerate(five_emails):
    result = classify_v0(email)
    print(f"\nEmail {i+1}: {email['subject']}")
    print(f"Expected : {email['correct_label']}")
    print(f"Got      : {result}")
    print(f"Length   : {len(result)} characters")
    print("-" * 60)
    time.sleep(1)

# What you should see:
# - Some return 1 word (lucky)
# - Some return 100+ word essays (broken)
# - No consistency in format
# - No consistency in categories used

V0 NAIVE CLASSIFIER — WEDNESDAY MORNING CHAOS

Email 1: Invoice Discrepancy
Expected : Billing
Got      : This email can be classified as a **customer support request** or a **billing issue inquiry**.
Length   : 94 characters
------------------------------------------------------------

Email 2: Login Issues
Expected : Technical
Got      : This email can be classified as a **technical support request** or **customer service inquiry**.
Length   : 96 characters
------------------------------------------------------------

Email 3: Integration with Slack
Expected : Feature Request
Got      : The email can be classified as a "Request" or "Integration Inquiry."
Length   : 68 characters
------------------------------------------------------------

Email 4: Is the System Down?
Expected : Technical
Got      : The email can be classified as a **Request for Technical Support**.
Length   : 67 characters
------------------------------------------------------------

Email 5: API Limits Inquiry
Expe

---
## Task 2 — Build V4 Structured
### Add all 4 components — one at a time

Watch how each addition improves the output.

In [6]:
# ── COMPONENT A: Role only ──
def classify_v1_role(email):
    prompt = f"""You are an expert support email classifier 
for a SaaS product company.

Classify this email: {email['body']}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()


# Test on email 1
email = five_emails[0]
result_v0 = classify_v0(email)
time.sleep(0.5)
result_v1 = classify_v1_role(email)

print(f"Email    : {email['subject']}")
print(f"Expected : {email['correct_label']}")
print()
print(f"V0 (no role) : {result_v0[:80]}")
print(f"V1 (role)    : {result_v1[:80]}")
print()
print("Notice: Role helps but format still inconsistent")

Email    : Invoice Discrepancy
Expected : Billing

V0 (no role) : The email can be classified as a "Customer Service Issue" or "Invoice Dispute."
V1 (role)    : Classification: Billing Issue

Notice: Role helps but format still inconsistent


In [7]:
# ── COMPONENT B: Role + Instruction ──
def classify_v2_instruction(email):
    prompt = f"""You are an expert support email classifier 
for a SaaS product company.

Classify the email into EXACTLY ONE of these categories:
- Billing
- Technical
- Feature Request
- Spam
- Other

Email: {email['body']}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()


time.sleep(0.5)
result_v2 = classify_v2_instruction(email)

print(f"Email    : {email['subject']}")
print(f"Expected : {email['correct_label']}")
print()
print(f"V1 (role only)         : {result_v1[:60]}")
print(f"V2 (role + instruction): {result_v2[:60]}")
print()
print("Notice: Now uses OUR categories. But format still varies.")

Email    : Invoice Discrepancy
Expected : Billing

V1 (role only)         : Classification: Billing Issue
V2 (role + instruction): Billing

Notice: Now uses OUR categories. But format still varies.


In [8]:
# ── COMPONENT C: Role + Instruction + Input (structured) ──
def classify_v3_input(email):
    prompt = f"""You are an expert support email classifier 
for a SaaS product company.

Classify the email into EXACTLY ONE of these categories:
- Billing
- Technical
- Feature Request
- Spam
- Other

EMAIL TO CLASSIFY:
Subject : {email['subject']}
From    : {email['sender']}
Body    : {email['body']}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()


time.sleep(0.5)
result_v3 = classify_v3_input(email)

print(f"V2 (no subject/sender) : {result_v2[:60]}")
print(f"V3 (with subject)      : {result_v3[:60]}")
print()
print("Notice: Subject + sender = extra signals for spam detection")

V2 (no subject/sender) : Billing
V3 (with subject)      : Billing

Notice: Subject + sender = extra signals for spam detection


In [9]:
# ── COMPONENT D: All 4 — Role + Instruction + Input + Output Format ──

CLASSIFY_PROMPT = """CRITICAL: Return ONLY the category name. Nothing else.

You are an expert support email classifier for a B2B SaaS company.

Classify the email into EXACTLY ONE of:
- Billing
- Technical
- Feature Request
- Spam
- Other

Rules:
- Return ONLY the category name. No explanation. No punctuation.
- Sarcastic billing complaints → still Billing
- Login broken because payment failed → Billing
- If unsure → Other

EMAIL TO CLASSIFY:
Subject : {subject}
From    : {sender}
Body    : {body}

REMINDER: Return ONLY the category name."""


def classify_v4(email):
    prompt = CLASSIFY_PROMPT.format(
        subject=email['subject'],
        sender=email['sender'],
        body=email['body']
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()


# Run V4 on all 5 emails
print("=" * 60)
print("V4 STRUCTURED — ALL 4 COMPONENTS")
print("=" * 60)

correct = 0
for i, email in enumerate(five_emails):
    result = classify_v4(email)
    is_correct = result == email['correct_label']
    if is_correct:
        correct += 1
    status = "✅" if is_correct else "❌"
    print(f"{status} Email {i+1}: Expected {email['correct_label']:16} Got {result}")
    time.sleep(0.5)

print(f"\nScore: {correct}/5 = {correct*20}%")

V4 STRUCTURED — ALL 4 COMPONENTS
✅ Email 1: Expected Billing          Got Billing
✅ Email 2: Expected Technical        Got Technical
✅ Email 3: Expected Feature Request  Got Feature Request
✅ Email 4: Expected Technical        Got Technical
❌ Email 5: Expected Technical        Got Other

Score: 4/5 = 80%


In [10]:
# ── FINAL COMPARISON — V0 vs V4 ──
print("=" * 60)
print("SAME EMAIL | SAME MODEL | V0 vs V4")
print("=" * 60)

email = five_emails[3]   # medium difficulty — sarcasm
print(f"Email    : {email['subject']}")
print(f"Body     : {email['body'][:100]}...")
print(f"Expected : {email['correct_label']}")
print()

v0_result = classify_v0(email)
time.sleep(1)
v4_result = classify_v4(email)

print(f"V0 Output ({len(v0_result)} chars):")
print(f"  {v0_result[:150]}")
print()
print(f"V4 Output ({len(v4_result)} chars):")
print(f"  {v4_result}")
print()
print("Lesson: Same model. Same email. PROMPT makes all the difference.")

SAME EMAIL | SAME MODEL | V0 vs V4
Email    : Is the System Down?
Body     : Something is broken in the system. Can you get it fixed?...
Expected : Technical

V0 Output (85 chars):
  This email can be classified as a "Request for Support" or "Technical Issue Inquiry."

V4 Output (9 chars):
  Technical

Lesson: Same model. Same email. PROMPT makes all the difference.


---
## Task 3 — temperature=0 Effect
### What happens without it?

In [ ]:
# Same ambiguous email — run 3 times
# With temperature=0 vs without

ambiguous_email = five_emails[4]   # hard email
print(f"Email: {ambiguous_email['subject']}")
print(f"Body : {ambiguous_email['body'][:100]}")
print()

# ── WITH temperature=0 (deterministic) ──
print("WITH temperature=0 (3 runs):")
for run in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": CLASSIFY_PROMPT.format(
            subject=ambiguous_email['subject'],
            sender=ambiguous_email['sender'],
            body=ambiguous_email['body']
        )}],
        temperature=0   # deterministic
    )
    result = response.choices[0].message.content.strip()
    print(f"  Run {run+1}: {result}")
    time.sleep(0.5)

print()

# ── WITHOUT temperature (default = 1.0) ──
print("WITHOUT temperature=0 (3 runs):")
for run in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": CLASSIFY_PROMPT.format(
            subject=ambiguous_email['subject'],
            sender=ambiguous_email['sender'],
            body=ambiguous_email['body']
        )}]
        # no temperature = default 1.0 = random
    )
    result = response.choices[0].message.content.strip()
    print(f"  Run {run+1}: {result}")
    time.sleep(0.5)

print()
print("Lesson:")
print("  temperature=0   → same answer every time → production safe")
print("  temperature=1.0 → different answers → unpredictable")
print("  For classifiers: ALWAYS use temperature=0")

---
## Task 4 — Add Confidence Score (5th Component)
### Make the classifier tell you when it's unsure

In [ ]:
# 5th component: Confidence Score
# Model returns: Category | Score
# This is the foundation of Self-Consistency (Session 4)

CONFIDENCE_PROMPT = """CRITICAL: Return ONLY in format: Category | Score

You are an expert support email classifier for a B2B SaaS company.

Classify the email into EXACTLY ONE of:
- Billing
- Technical
- Feature Request
- Spam
- Other

Also give a Confidence Score 1-10:
- 10 = absolutely sure
- 5  = could go either way
- 1  = total guess

Rules:
- Return ONLY: Category | Score
- Example: Billing | 9
- Example: Technical | 6
- No explanation. Nothing else.

EMAIL:
Subject : {subject}
From    : {sender}
Body    : {body}"""


def classify_with_confidence(email):
    prompt = CONFIDENCE_PROMPT.format(
        subject=email['subject'],
        sender=email['sender'],
        body=email['body']
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    raw = response.choices[0].message.content.strip()

    # Parse: "Billing | 8" → category, confidence
    parts = raw.split("|")
    category   = parts[0].strip()
    confidence = int(parts[1].strip()) if len(parts) > 1 else 0

    return category, confidence


# Run on all 5 emails
print("=" * 60)
print("CLASSIFIER WITH CONFIDENCE SCORE")
print("=" * 60)

for email in five_emails:
    category, confidence = classify_with_confidence(email)
    expected  = email['correct_label']
    is_correct = category == expected
    status    = "✅" if is_correct else "❌"

    # Flag low confidence for human review
    flag = "⚠️ REVIEW" if confidence <= 5 else ""

    print(f"{status} [{email['difficulty']:6}] "
          f"Expected: {expected:16} "
          f"Got: {category:16} "
          f"Confidence: {confidence}/10 {flag}")
    time.sleep(0.5)

print()
print("Key Insight:")
print("  Easy emails   → confidence 8-10 → auto route")
print("  Medium emails → confidence 6-7  → double check")
print("  Hard emails   → confidence 1-5  → human review")
print()
print("This confidence score is the foundation of")
print("Self-Consistency in Session 4!")

---
## Chapter 3 Summary

| Component | What it does | Without it |
|-----------|-------------|------------|
| **Role** | Sets expertise & persona | Generic, unfocused answers |
| **Instruction** | Exact task + categories | Model invents own categories |
| **Input** | Subject + sender + body | Misses spam signals in subject |
| **Output Format** | One word, no explanation | 120-word essays |
| **temperature=0** | Deterministic output | Different answer every run |

### What's Next?
Chapter 4 — **Prompt Structuring**  
Our prompt works but as requirements grow — urgency levels, sub-categories, multi-language — it becomes a wall of text. We'll structure it like a specification document.